# 02 — Limpeza de Dados

**Objetivo:** transformar o DataFrame bruto em algo confiável para análise.
**Entrada:** DataFrame de `carregar_dados()` — tudo como string, formato wide.
**Saída:** DataFrame limpo, tipos corretos, formato long.
**Próximo passo:** copiar o consolidado para `limpar_dados()` em `pipeline.py`.

## Célula 1 — Setup

Reutilizamos `carregar_dados()` do pipeline para não duplicar código.
O notebook de limpeza sempre começa do dado bruto — nunca do meio do processo.

In [1]:
import sys
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ))

from pipeline import carregar_dados, COLUNAS_ID, COLUNAS_VIOLENCIA

ARQUIVO = RAIZ / "data" / "raw" / "BaseDPEvolucaoMensalCisp.csv"

df_bruto = carregar_dados(ARQUIVO)
print("Bruto:", df_bruto.shape)
df_bruto.head(2)

2026-05-11 18:51:29 [INFO] Dados carregados: 37588 linhas, 65 colunas


Bruto: (37588, 65)


,cisp,mes,ano,mes_ano,aisp,risp,munic,mcirc,regiao,hom_doloso,...,cmp,cmba,ameaca,pessoas_desaparecidas,encontro_cadaver,encontro_ossada,pol_militares_mortos_serv,pol_civis_mortos_serv,registro_ocorrencias,fase
0,1,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,0,...,NaN,NaN,21,2,0,0,0,0,578,3
1,4,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,15,6,0,1,0,0,441,3


## Célula 2 — Sempre copiar primeiro

Antes de qualquer transformação, criamos uma cópia.
Isso protege `df_bruto` — se errar em qualquer etapa, é só rodar esta célula de novo
sem precisar recarregar o arquivo inteiro.

Sem `.copy()`, o pandas pode modificar o original sem avisar (SettingWithCopyWarning).

In [2]:
df = df_bruto.copy()

# Confirmando que são objetos diferentes na memória
print("São o mesmo objeto?", df is df_bruto)

São o mesmo objeto? False


## Célula 3 — Normalizar strings

Colunas de texto podem ter espaços sobrando ou capitalização inconsistente.
Isso causa problemas silenciosos no `groupby` — `"capital"` e `"Capital"` viram grupos diferentes.

`.str` é o acessor de string do pandas — permite aplicar métodos de string em toda a coluna de uma vez.
`.strip()` remove espaços no início e no fim.
`.upper()` converte para maiúsculas.

In [3]:
for col in ["munic", "regiao", "aisp", "cisp"]:
    df[col] = df[col].str.strip().str.upper()

print("Valores únicos de regiao após normalização:")
print(df["regiao"].unique())

Valores únicos de regiao após normalização:
<StringArray>
[                      'CAPITAL',            'BAIXADA FLUMINENSE',
                      'INTERIOR', 'GRANDE NITERÃÂÃÂÃÂÃÂ³I',
                'GRANDE NITERÓI']
Length: 5, dtype: str


## Célula 4 — Converter mes_ano para data

O formato do arquivo é `"2003m01"` — uma string, não uma data.
Precisamos convertê-la para o pandas entender como tempo (ordenação, agrupamento por mês etc).

`pd.to_datetime()` converte strings para datas.
`format="%Ym%m"` diz como ler: `%Y` = ano com 4 dígitos, `m` = letra m literal, `%m` = mês com 2 dígitos.
`errors="coerce"` transforma valores inválidos em NaT (Not a Time) em vez de travar.
`.dt.to_period("M")` converte para período mensal — mais adequado que datetime para dados mensais.

In [4]:
df["mes_ano"] = pd.to_datetime(df["mes_ano"], format="%Ym%m", errors="coerce")
df["mes_ano"] = df["mes_ano"].dt.to_period("M")

print("Tipo da coluna mes_ano:", df["mes_ano"].dtype)
print("Primeiros valores:")
print(df["mes_ano"].unique()[:8])

Tipo da coluna mes_ano: period[M]
Primeiros valores:
<PeriodArray>
['2003-01', '2003-02', '2003-03', '2003-04', '2003-05', '2003-06', '2003-07',
 '2003-08']
Length: 8, dtype: period[M]


## Célula 5 — Converter colunas numéricas

Lemos tudo como string (`dtype=str`) — agora convertemos os crimes para inteiro.

`pd.to_numeric()` converte strings para números.
Usamos ele no lugar de `.astype(int)` porque `astype` trava se encontrar `""` ou texto inválido.
`errors="coerce"` transforma valores inválidos em `NaN` em vez de travar.

Após a conversão, `NaN` vira `0` com `.fillna(0)` — neste dataset, ausência de registro
significa zero ocorrências, não dado faltante.

In [5]:
for col in COLUNAS_VIOLENCIA:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

print("Tipos após conversão:")
print(df[COLUNAS_VIOLENCIA].dtypes)
print()
print("Amostra dos valores:")
print(df[COLUNAS_VIOLENCIA].head(3))

Tipos após conversão:
hom_doloso             int64
latrocinio             int64
cvli                   int64
letalidade_violenta    int64
dtype: object

Amostra dos valores:
   hom_doloso  latrocinio  cvli  letalidade_violenta
0           0           0     0                    0
1           3           0     3                    3
2           3           0     3                    3


## Célula 6 — melt(): wide → long

Essa é a maior transformação estrutural da limpeza.

**Wide (atual):** cada crime é uma coluna
```
cisp | mes_ano | aisp | hom_doloso | latrocinio | cvli
1    | 2003-01 | 5    | 0          | 2          | 2
```

**Long (resultado):** cada crime vira uma linha
```
cisp | mes_ano | aisp | tipo_crime  | qtd_ocorrencias
1    | 2003-01 | 5    | hom_doloso  | 0
1    | 2003-01 | 5    | latrocinio  | 2
1    | 2003-01 | 5    | cvli        | 2
```

`id_vars` = colunas que ficam fixas (o contexto de cada linha)
`value_vars` = colunas que viram linhas
`var_name` = nome da nova coluna com os nomes dos crimes
`value_name` = nome da nova coluna com os valores

In [6]:
df = df.melt(
    id_vars=COLUNAS_ID,
    value_vars=COLUNAS_VIOLENCIA,
    var_name="tipo_crime",
    value_name="qtd_ocorrencias",
)

print("Shape após melt:", df.shape)
df.head(6)

Shape após melt: (150352, 8)


,cisp,mes_ano,aisp,risp,munic,regiao,tipo_crime,qtd_ocorrencias
0,1,2003-01,5,1,RIO DE JANEIRO,CAPITAL,hom_doloso,0
1,4,2003-01,5,1,RIO DE JANEIRO,CAPITAL,hom_doloso,3
2,5,2003-01,5,1,RIO DE JANEIRO,CAPITAL,hom_doloso,3
3,6,2003-01,1,1,RIO DE JANEIRO,CAPITAL,hom_doloso,6
4,7,2003-01,1,1,RIO DE JANEIRO,CAPITAL,hom_doloso,4
5,9,2003-01,2,1,RIO DE JANEIRO,CAPITAL,hom_doloso,1


## Célula 7 — Remover duplicatas

`drop_duplicates()` remove linhas 100% idênticas.
Passamos `subset` para checar apenas as colunas que identificam um registro único —
a combinação de delegacia + período + crime não deveria se repetir.

In [7]:
antes = df.shape[0]
df = df.drop_duplicates(subset=["cisp", "mes_ano", "tipo_crime"])
depois = df.shape[0]

print(f"Linhas removidas: {antes - depois}")
print("Shape final:", df.shape)

Linhas removidas: 0
Shape final: (150352, 8)


## Célula 8 — Validação final

Antes de passar o DataFrame adiante, conferimos se o resultado faz sentido:
tipos corretos, sem nulos nas colunas críticas, e uma amostra visual.

In [8]:
print("Tipos finais:")
print(df.dtypes)
print()
print("Nulos por coluna:")
print(df.isnull().sum())
print()
print("Crimes disponíveis:")
print(df["tipo_crime"].unique())

Tipos finais:
cisp                     str
mes_ano            period[M]
aisp                     str
risp                     str
munic                    str
regiao                   str
tipo_crime               str
qtd_ocorrencias        int64
dtype: object

Nulos por coluna:
cisp               0
mes_ano            0
aisp               0
risp               0
munic              0
regiao             0
tipo_crime         0
qtd_ocorrencias    0
dtype: int64

Crimes disponíveis:
<StringArray>
['hom_doloso', 'latrocinio', 'cvli', 'letalidade_violenta']
Length: 4, dtype: str


---
## Consolidado — o que vai para `limpar_dados()` no pipeline.py

```python
def limpar_dados(df):
    df = df.copy()

    for col in ["munic", "regiao", "aisp", "cisp"]:
        df[col] = df[col].str.strip().str.upper()

    df["mes_ano"] = pd.to_datetime(df["mes_ano"], format="%Ym%m", errors="coerce")
    df["mes_ano"] = df["mes_ano"].dt.to_period("M")

    for col in COLUNAS_VIOLENCIA:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    df = df.melt(
        id_vars=COLUNAS_ID,
        value_vars=COLUNAS_VIOLENCIA,
        var_name="tipo_crime",
        value_name="qtd_ocorrencias",
    )

    df = df.drop_duplicates(subset=["cisp", "mes_ano", "tipo_crime"])

    log.info("Limpeza concluida. Shape final: %s", df.shape)
    return df
```